In [1]:
!pip install pyspark

In [5]:
# Initiate PySpark session
from pyspark.sql import SparkSession

# Create a Spark Session
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Colab_PySpark_MapReduce") \
    .getOrCreate()

# Check to see if it worked
spark.getActiveSession()

In [10]:
# Download wordcount.txt
!wget https://github.com/nivdul/spark-in-practice-scala/blob/master/data/wordcount.txt

--2026-04-07 14:45:43--  https://github.com/nivdul/spark-in-practice-scala/blob/master/data/wordcount.txt
Resolving github.com (github.com)... 20.27.177.113
Connecting to github.com (github.com)|20.27.177.113|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘wordcount.txt’

wordcount.txt           [ <=>                ] 247.46K  --.-KB/s    in 0.1s    

2026-04-07 14:45:44 (1.75 MB/s) - ‘wordcount.txt’ saved [253402]



In [14]:
inputRDD = spark.sparkContext.textFile("wordcount.txt")
print(inputRDD)


wordcount.txt MapPartitionsRDD[7] at textFile at NativeMethodAccessorImpl.java:0


In [15]:
print('The number of partitions: ',inputRDD.getNumPartitions(), '\nThe total number of elements: ', inputRDD.count())


The number of partitions:  2 
The total number of elements:  1499


In [16]:
# Contoh map-reduce
inputRDD.mapPartitions(lambda m: [1]).reduce(lambda a,b: a+b)

2

In [20]:
# Map setiap kata menjadi satu nilai=1, kemudian tambahkan satu-persatu sampai akhir = total jumlah kata
inputRDD.map(lambda m: 1).reduce(lambda a,b: a+b)

1499

In [24]:
# MapReduce untuk menghitung jumlah kemunculan setiap kata

# Mapper
words = inputRDD.flatMap(lambda x: x.split(' '))
print(words)

wordsOne = words.map(lambda x: (x, 1))
print(wordsOne)

PythonRDD[17] at RDD at PythonRDD.scala:56
PythonRDD[18] at RDD at PythonRDD.scala:56


In [25]:
# Reduce
from operator import add

wordCounts = wordsOne.reduceByKey(add)
print(wordCounts)

PythonRDD[23] at RDD at PythonRDD.scala:56


In [27]:
# Pengumpulan hasil
output = wordCounts.collect()
output[:10]

[('', 5688),
 ('html>', 1),
 ('<html', 1),
 ('data-color-mode="auto"', 1),
 ('data-dark-theme="dark"', 1),
 ('data-a11y-animated-images="system"', 1),
 ('data-a11y-link-underlines="true"', 1),
 ('>', 23),
 ('<head>', 1),
 ('<link', 36)]

In [28]:
# Tampilkan top 10

sorted(output, key=lambda x: x[1], reverse = True)[0:10]

[('', 5688),
 ('0', 2991),
 ('1', 776),
 ('data-view-component="true"', 175),
 ('1.75', 155),
 ('aria-hidden="true"', 142),
 ('16', 134),
 ('viewBox="0', 129),
 ('class="octicon', 129),
 ('crossorigin="anonymous"', 128)]

In [29]:
import kagglehub, os

os.environ['KAGGLEHUB_CACHE'] = "/content/kaggle"
# Download fish dataset
path = kagglehub.dataset_download("vipullrathod/fish-market")

print("Path to dataset files:", path)

100%|██████████| 2.38k/2.38k [00:00<00:00, 6.75MB/s]

Extracting files...
Path to dataset files: /content/kaggle/datasets/vipullrathod/fish-market/versions/1


In [32]:
import pyspark.sql.functions as F

# 1. Initialize Spark session
spark = SparkSession.builder.appName("HeaviestFish").getOrCreate()

# 2. Load the fish.csv
df = spark.read.csv(path + "/Fish.csv", header=True, inferSchema=True)

# 3. Cari ikan terberat dengan groupby
max_weight_per_species = df.groupBy("Species").agg(F.max("Weight").alias("MaxWeight"))

# 4. tampilkan hasil
max_weight_per_species.show()

+---------+---------+
|  Species|MaxWeight|
+---------+---------+
|    Roach|    390.0|
|    Smelt|     19.9|
|   Parkki|    300.0|
|Whitefish|   1000.0|
|     Pike|   1650.0|
|    Bream|   1000.0|
|    Perch|   1100.0|
+---------+---------+



In [35]:
# TUGAS
# Cari ikan terberat per spesies menggunakan map-reduce seperti di contoh wordcount.
# Tips: load dataframe (df) ke spark rdd terlebih dahulu